In [32]:
import boto3
import pandas as pd
import json
import io
import seaborn as sns

In [2]:
# Connexion à MinIO
s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
    region_name="us-east-1"
)

In [4]:
resp = s3.list_objects_v2(Bucket="datalake", Prefix="bronze/date=")
print([o["Key"] for o in resp.get("Contents", [])])

['bronze/date=2026-03-12/dpe_034335_0000.json', 'bronze/date=2026-03-12/dpe_034436_0001.json']


In [5]:
# Lecture du fichierr
obj = s3.get_object(Bucket="datalake", Key="bronze/date=2026-03-12/dpe_034335_0000.json")
lines = obj["Body"].read().decode("utf-8").strip().split("\n")
df = pd.DataFrame([json.loads(l) for l in lines])

In [15]:
#df.info() + df.isnull().sum()
df.columns

Index(['configuration_installation_chauffage_n2',
       'configuration_installation_chauffage_n1',
       'conso_chauffage_installation_chauffage_n1',
       'type_generateur_n1_ecs_n1', 'numero_voie_ban', 'score_ban',
       'conso_chauffage_generateur_n1_installation_n2',
       'surface_habitable_immeuble', 'conso_auxiliaires_ep',
       'conso_chauffage_installation_chauffage_n2',
       ...
       'emission_ges_5_usages_energie_n3', 'emission_ges_ecs_energie_n3',
       'emission_ges_chauffage_energie_n3',
       'conso_chauffage_generateur_n2_installation_n1',
       'cout_chauffage_energie_n3', 'position_logement_dans_immeuble',
       'typologie_logement', 'nom_residence', 'cop_generateur_n1_ecs_n1',
       'electricite_pv_autoconsommee'],
      dtype='object', length=211)

In [26]:
selection = ["etiquette_dpe", "etiquette_ges", "surface_habitable_logement", "type_batiment", "periode_construction", "type_energie_principale_chauffage", "code_postal_ban", "date_reception_dpe", "conso_5_usages_par_m2_ef"]
df_selected = df[selection].copy()
df_selected.columns

Index(['etiquette_dpe', 'etiquette_ges', 'surface_habitable_logement',
       'type_batiment', 'periode_construction',
       'type_energie_principale_chauffage', 'code_postal_ban',
       'date_reception_dpe', 'conso_5_usages_par_m2_ef'],
      dtype='object')

In [27]:
df_selected.head()

,etiquette_dpe,etiquette_ges,surface_habitable_logement,type_batiment,periode_construction,type_energie_principale_chauffage,code_postal_ban,date_reception_dpe,conso_5_usages_par_m2_ef
0,C,A,NaN,immeuble,2001-2005,Électricité,11000,2026-01-07,73.0
1,C,C,49.4,appartement,2013-2021,Gaz naturel,06530,2026-01-05,61.0
2,C,A,88.7,maison,1978-1982,Électricité,33490,2026-01-05,74.0
3,C,C,18.5,appartement,1989-2000,Réseau de Chauffage urbain,92800,2026-01-06,204.0
4,C,C,100.0,appartement,1975-1977,Réseau de Chauffage urbain,91940,2026-01-08,129.0


In [28]:
df_selected.shape
df_selected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 9 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   etiquette_dpe                      201 non-null    object 
 1   etiquette_ges                      201 non-null    object 
 2   surface_habitable_logement         192 non-null    float64
 3   type_batiment                      201 non-null    object 
 4   periode_construction               201 non-null    object 
 5   type_energie_principale_chauffage  201 non-null    object 
 6   code_postal_ban                    173 non-null    object 
 7   date_reception_dpe                 201 non-null    object 
 8   conso_5_usages_par_m2_ef           201 non-null    float64
dtypes: float64(2), object(7)
memory usage: 14.3+ KB


In [29]:
df_selected.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
etiquette_dpe,201,5,C,107,NaN,NaN,NaN,NaN,NaN,NaN,NaN
etiquette_ges,201,6,C,86,NaN,NaN,NaN,NaN,NaN,NaN,NaN
surface_habitable_logement,192.0,NaN,NaN,NaN,74.689583,34.536068,9.8,54.25,73.45,90.45,245.4
type_batiment,201,3,appartement,120,NaN,NaN,NaN,NaN,NaN,NaN,NaN
periode_construction,201,10,1948-1974,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type_energie_principale_chauffage,201,6,Gaz naturel,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
code_postal_ban,173,81,24000,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_reception_dpe,201,31,2026-01-05,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
conso_5_usages_par_m2_ef,201.0,NaN,NaN,NaN,141.741294,74.689176,39.0,83.0,123.0,179.0,370.7


In [30]:
df_selected.isna().sum().sort_values(ascending=False)

code_postal_ban                      28
surface_habitable_logement            9
etiquette_dpe                         0
type_batiment                         0
etiquette_ges                         0
periode_construction                  0
type_energie_principale_chauffage     0
date_reception_dpe                    0
conso_5_usages_par_m2_ef              0
dtype: int64

In [35]:
#corr = df_selected.select_dtypes(include="number").corr()
#sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")